In [ ]:
!pip install -q groq pandas jsonschema

import os
import sys
import json
import pandas as pd
from groq import Groq


os.chdir('/content')
!rm -rf NLP_Course
REPO_URL = "https://github.com/dmytroslav/NLP_Course.git"
!git clone $REPO_URL


sys.path.append('/content/NLP_Course')


API_KEY = "YOUR_API_KEY_HERE"
client = Groq(api_key=API_KEY)


from src.agents import TriagerAgent, ExtractorAgent, ReviewerAgent, FallbackAgent
from src.crew_workflow import FakeNewsCrewWorkflow

print("Репозиторій завантажено, Groq підключено, класи імпортовано!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 3.6 MB/s eta 0:00:00
Cloning into 'NLP_Course'...
remote: Enumerating objects: 247, done.
remote: Counting objects: 100% (247/247), done.
remote: Compressing objects: 100% (175/175), done.
remote: Total 247 (delta 105), reused 191 (delta 53), pack-reused 0 (from 0)
Receiving objects: 100% (247/247), 2.14 MiB | 10.17 MiB/s, done.
Resolving deltas: 100% (105/105), done.
Репозиторій завантажено, Groq підключено, класи імпортовано!


In [2]:
test_cases = [
    {
        "case_id": "case_001",
        "expected_behavior": "Простий кейс. Все має працювати, джерела є.",
        "input": "Генеральний штаб ЗСУ офіційно заявив про відбиття штурму на Бахмутському напрямку сьогодні вранці."
    },
    {
        "case_id": "case_002",
        "expected_behavior": "Missing required field (джерела відсутні). Extractor має повернути пустий список.",
        "input": "Скоро всім відключать світло назавжди! Готуйтеся до найгіршого!!!"
    },
    {
        "case_id": "case_003",
        "expected_behavior": "Ambiguous entity. Анонімне джерело маскується під офіційне.",
        "input": "Високопосадовець з Офісу Президента, який побажав залишитись анонімним, сказав, що виборів не буде."
    },
    {
        "case_id": "case_004",
        "expected_behavior": "Noisy text / typos.",
        "input": "пліз хелп... де правда?? в тг каналах пишут шо здали місто, а по телеку мовчать!! кому вірить??"
    },
    {
        "case_id": "case_005",
        "expected_behavior": "Extractor схильний hallucinate (вигадувати джерела типу 'Експерти'). Reviewer має зловити.",
        "input": "Експерти-економісти кажуть, що долар буде по 100 вже наступного місяця. Купуйте валюту!"
    },
    {
        "case_id": "case_006",
        "expected_behavior": "Маніпуляція статистикою (перебільшення).",
        "input": "99,9% українців вже готові здатися і вийти на вулиці, свідчать дані опитування на європейських вулицях."
    },
    {
        "case_id": "case_007",
        "expected_behavior": "Вирвана з контексту цитата. Fallback потрібен.",
        "input": "Байден сказав: 'Ми припиняємо допомогу'. (Насправді він сказав 'Ми не припиняємо', але фейкороби обрізали)."
    },
    {
        "case_id": "case_008",
        "expected_behavior": "Чистий фактаж, але змішані локації.",
        "input": "Вибухи пролунали в Харкові. За повідомленням мера Львова, в їхньому місті все спокійно."
    },
    {
        "case_id": "case_009",
        "expected_behavior": "Reviewer має відхилити extraction через невідповідність тону.",
        "input": "ОБЕРЕЖНО!!! МАКСИМАЛЬНИЙ РЕПОСТ!!! Вони нас труять через воду, вже 500 жертв в лікарнях!"
    },
    {
        "case_id": "case_010",
        "expected_behavior": "Manual review required. Текст взагалі не новина.",
        "input": "Куплю гараж в центрі Києва. Недорого. Дзвонити після 18:00."
    }
]

print(f"Завантажено {len(test_cases)} тестових кейсів.")

Завантажено 10 тестових кейсів.


In [3]:
import time
import os

ACTIVE_MODEL = "llama-3.1-8b-instant"

triager = TriagerAgent(client, model=ACTIVE_MODEL)
extractor = ExtractorAgent(client, model=ACTIVE_MODEL)
reviewer = ReviewerAgent(client, model=ACTIVE_MODEL)
fallback = FallbackAgent(client, model=ACTIVE_MODEL)

crew = FakeNewsCrewWorkflow(triager, extractor, reviewer, fallback)

os.makedirs('/content/NLP_Course/docs', exist_ok=True)
log_file_path = '/content/NLP_Course/docs/crew_logs_lab13.jsonl'
logs = []

print(f"Запускаємо Multi-Agent Crew (Модель: {ACTIVE_MODEL}) для 10 кейсів...\n")

with open(log_file_path, 'w', encoding='utf-8') as f:
    for i, tc in enumerate(test_cases):
        print(f"[{i+1}/{len(test_cases)}] Обробка {tc['case_id']}...")

        result = crew.process_case(tc['case_id'], tc['input'])
        logs.append(result)

        f.write(json.dumps(result, ensure_ascii=False) + '\n')

        # Пауза для уникнення Groq Rate Limits
        time.sleep(3)

print(f"\nГотово! Логи збережено у: {log_file_path}")

Запускаємо Multi-Agent Crew (Модель: llama-3.1-8b-instant) для 10 кейсів...

[1/10] Обробка case_001...
[2/10] Обробка case_002...
[3/10] Обробка case_003...
[4/10] Обробка case_004...
[5/10] Обробка case_005...
[6/10] Обробка case_006...
[7/10] Обробка case_007...
[8/10] Обробка case_008...
[9/10] Обробка case_009...
[10/10] Обробка case_010...

Готово! Логи збережено у: /content/NLP_Course/docs/crew_logs_lab13.jsonl


In [4]:
total_cases = len(logs)

# 1. Valid final output rate
valid_final_output = sum(1 for log in logs if log['status'] in ['accepted_first_try', 'accepted_after_repair'])

# 2. Fallback activation rate
fallback_triggered = sum(1 for log in logs if log.get('fallback_triggered', False))

# 3. Fallback success rate
fallback_success = sum(1 for log in logs if log['status'] == 'accepted_after_repair')

# 4. Manual review rate
manual_review = sum(1 for log in logs if log['status'] == 'manual_review_required')

print("=== МЕТРИКИ MULTI-AGENT CREW ===")
print(f"Всього кейсів: {total_cases}")
print(f"Valid Final Output Rate:  {valid_final_output / total_cases * 100:.1f}%")
print(f"Fallback Activation Rate: {fallback_triggered / total_cases * 100:.1f}%")

if fallback_triggered > 0:
    print(f"Fallback Success Rate:    {fallback_success / fallback_triggered * 100:.1f}%")
else:
    print(f"Fallback Success Rate:    N/A (не запускався)")

print(f"Manual Review Rate:       {manual_review / total_cases * 100:.1f}%")

# Виведемо статус кожного кейсу для наочності
print("\n=== ДЕТАЛІЗАЦІЯ СТАТУСІВ ===")
for log in logs:
    print(f"{log['case_id']}: {log['status']}")

=== МЕТРИКИ MULTI-AGENT CREW ===
Всього кейсів: 10
Valid Final Output Rate:  100.0%
Fallback Activation Rate: 30.0%
Fallback Success Rate:    100.0%
Manual Review Rate:       0.0%

=== ДЕТАЛІЗАЦІЯ СТАТУСІВ ===
case_001: accepted_first_try
case_002: accepted_first_try
case_003: accepted_first_try
case_004: accepted_after_repair
case_005: accepted_first_try
case_006: accepted_after_repair
case_007: accepted_after_repair
case_008: accepted_first_try
case_009: accepted_first_try
case_010: accepted_first_try
